# Whisper speech recognition: video folder -> Elasticsearch NDJSON file

This Kaggle-ready notebook takes a folder of videos, runs Whisper speech recognition across dual GPUs (Tesla T4 x2), and writes individual JSON outputs.

## 1. Install libraries

Run this cell on Kaggle if the environment does not already have the needed packages.

In [1]:
%pip install -U faster-whisper tqdm av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.9 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Note: you may need to

## 2. Config

Set `VIDEO_DIR` to your Kaggle video folder. `OUTPUT_DIR` stores the individual video transcript JSON files.

In [2]:
from pathlib import Path

VIDEO_DIR = Path("/kaggle/input/datasets/nguynthytrn/aic-batch2-part2/Videos_M06")
OUTPUT_DIR = Path("/kaggle/working/whisper_json_outputs")
ES_INDEX = "aic_whisper_segments"

MODEL_SIZE = "large-v3-turbo" # Recommended model size
DEVICE = "cuda"               # CUDA GPU execution
COMPUTE_TYPE = "float16"        # FP16 precision
LANGUAGE = "vi"                # Forced Vietnamese language recognition

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v"}

## 3. Import and find videos

In [3]:
import hashlib
import json
from datetime import datetime, timezone

from tqdm.auto import tqdm

def find_videos(video_dir: Path):
    video_dir = video_dir.expanduser().resolve()
    if not video_dir.exists():
        raise FileNotFoundError(f"Video folder not found: {video_dir}")
    return sorted(
        p for p in video_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )

videos = find_videos(VIDEO_DIR)
print(f"Found {len(videos)} videos in {VIDEO_DIR.resolve()}")
videos[:5]

Found 30 videos in /kaggle/input/datasets/nguynthytrn/aic-batch2-part2/Videos_M06


[PosixPath('/kaggle/input/datasets/nguynthytrn/aic-batch2-part2/Videos_M06/videos/M06_V001.mp4'),
 PosixPath('/kaggle/input/datasets/nguynthytrn/aic-batch2-part2/Videos_M06/videos/M06_V002.mp4'),
 PosixPath('/kaggle/input/datasets/nguynthytrn/aic-batch2-part2/Videos_M06/videos/M06_V003.mp4'),
 PosixPath('/kaggle/input/datasets/nguynthytrn/aic-batch2-part2/Videos_M06/videos/M06_V004.mp4'),
 PosixPath('/kaggle/input/datasets/nguynthytrn/aic-batch2-part2/Videos_M06/videos/M06_V005.mp4')]

## 4. Load Whisper model (Dual GPU Setup)

CTranslate2 natively supports data parallelism across multiple GPUs by passing a list of device indices.

In [4]:
import torch
from faster_whisper import WhisperModel

# Dynamically detect available GPUs
if DEVICE == "cuda" and torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    GPU_DEVICES = list(range(num_gpus))
else:
    num_gpus = 0
    GPU_DEVICES = [0]

NUM_WORKERS = max(num_gpus, 1)

print(f"Detected {num_gpus} GPU(s): {GPU_DEVICES}")
print(f"Loading WhisperModel '{MODEL_SIZE}' on device_index={GPU_DEVICES} with {NUM_WORKERS} worker(s)...")

# Native multi-GPU initialization: CTranslate2 creates model replicas on all specified GPUs
model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE if num_gpus > 0 else "cpu",
    device_index=GPU_DEVICES if num_gpus > 0 else 0,
    compute_type=COMPUTE_TYPE,
    num_workers=NUM_WORKERS
)
print("[SUCCESS] WhisperModel initialized across GPUs successfully!")

Detected 2 GPU(s): [0, 1]
Loading WhisperModel 'large-v3-turbo' on device_index=[0, 1] with 2 worker(s)...


[SUCCESS] WhisperModel initialized across GPUs successfully!


## 5. Transcribe helper and Elasticsearch document format

In [5]:
import av

def stable_id(*parts: str) -> str:
    raw = "|".join(str(part) for part in parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

def has_audio_stream(video_path: Path) -> bool:
    """Check if media file contains at least one audio stream (prevents IndexError on silent CCTV/traffic videos)."""
    try:
        with av.open(str(video_path), mode="r", metadata_errors="ignore") as container:
            return len(container.streams.audio) > 0
    except Exception:
        return False

def segment_to_document(video_path: Path, segment, segment_index: int, detected_language: str):
    # Clean segment dictionary without duplicating root video metadata
    return {
        "segment_id": segment_index,
        "start": float(round(segment.start, 3)),
        "end": float(round(segment.end, 3)),
        "duration": float(round(segment.end - segment.start, 3)),
        "text": segment.text.strip(),
        "avg_logprob": float(round(segment.avg_logprob, 4)) if segment.avg_logprob is not None else None,
        "no_speech_prob": float(round(segment.no_speech_prob, 4)) if segment.no_speech_prob is not None else None,
    }

## 6. Run Whisper across 2 GPUs with Dynamic Scheduling

Videos are submitted to a thread pool connected to the multi-GPU model. As soon as any GPU is free, it immediately picks up the next video.

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import threading
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed

total_segments = 0
processed_videos = 0
errors = []
counter_lock = threading.Lock()

pbar = tqdm(total=len(videos), desc=f"Transcribing videos across {NUM_WORKERS} GPU worker(s)")

def process_single_video(video_path: Path):
    global total_segments, processed_videos
    
    try:
        # 1. Skip Whisper if video has no audio stream (CCTV, surveillance cameras, etc.)
        if not has_audio_stream(video_path):
            json_output_path = OUTPUT_DIR / f"{video_path.stem}.json"
            video_data = {
                "video_id": video_path.stem,
                "file_name": video_path.name,
                "video_path": str(video_path.resolve()),
                "language": "none",
                "total_segments": 0,
                "segments": []
            }
            with json_output_path.open("w", encoding="utf-8") as f:
                json.dump(video_data, f, ensure_ascii=False, indent=2)
            with counter_lock:
                processed_videos += 1
            return

        # 2. Transcribe via multi-GPU model (CTranslate2 automatically assigns free GPU worker)
        segments, info = model.transcribe(
            str(video_path),
            language=LANGUAGE,
            vad_filter=True,
            word_timestamps=False,
        )
        detected_language = getattr(info, "language", None) or LANGUAGE or "vi"
        
        video_docs = []
        for segment_index, segment in enumerate(segments):
            if segment.text and segment.text.strip():
                doc = segment_to_document(video_path, segment, segment_index, detected_language)
                if "_id" in doc:
                    doc.pop("_id")
                video_docs.append(doc)

        # Save individual JSON file named after the video stem
        json_output_path = OUTPUT_DIR / f"{video_path.stem}.json"
        video_data = {
            "video_id": video_path.stem,
            "file_name": video_path.name,
            "video_path": str(video_path.resolve()),
            "language": detected_language,
            "total_segments": len(video_docs),
            "segments": video_docs
        }
        with json_output_path.open("w", encoding="utf-8") as f:
            json.dump(video_data, f, ensure_ascii=False, indent=2)
            
        with counter_lock:
            total_segments += len(video_docs)
            processed_videos += 1

    except IndexError as exc:
        # Fallback if audio stream was missing or malformed
        json_output_path = OUTPUT_DIR / f"{video_path.stem}.json"
        video_data = {
            "video_id": video_path.stem,
            "file_name": video_path.name,
            "video_path": str(video_path.resolve()),
            "language": "none",
            "total_segments": 0,
            "segments": []
        }
        with json_output_path.open("w", encoding="utf-8") as f:
            json.dump(video_data, f, ensure_ascii=False, indent=2)
        with counter_lock:
            processed_videos += 1
    except Exception as exc:
        errors.append({"video_path": str(video_path), "error": repr(exc), "traceback": traceback.format_exc()})
    finally:
        pbar.update(1)

# Execute with dynamic scheduling across GPU workers
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(process_single_video, v_path) for v_path in videos]
    for future in as_completed(futures):
        pass

pbar.close()

print(f"[SUCCESS] Successfully processed {processed_videos} videos with {total_segments} total transcript segments")
print(f"[INFO] Individual JSON files saved in folder: {OUTPUT_DIR.resolve()}")
if errors:
    print(f"[WARNING] Encountered {len(errors)} errors during transcription.")
    print(f"[DEBUG] Sample Error [0]:\nVideo: {errors[0]['video_path']}\nError: {errors[0]['error']}")
    print(f"Traceback:\n{errors[0]['traceback']}")

Transcribing videos across 2 GPU worker(s):   0%|          | 0/30 [00:00<?, ?it/s]

[SUCCESS] Successfully processed 30 videos with 11416 total transcript segments
[INFO] Individual JSON files saved in folder: /kaggle/working/whisper_json_outputs


In [7]:
!zip -r M06.zip /kaggle/working/whisper_json_outputs
from IPython.display import FileLink
FileLink("/kaggle/working/M06.zip")

  adding: kaggle/working/whisper_json_outputs/ (stored 0%)
  adding: kaggle/working/whisper_json_outputs/M06_V020.json (deflated 81%)
  adding: kaggle/working/whisper_json_outputs/M06_V002.json (deflated 81%)
  adding: kaggle/working/whisper_json_outputs/M06_V009.json (deflated 84%)
  adding: kaggle/working/whisper_json_outputs/M06_V014.json (deflated 79%)
  adding: kaggle/working/whisper_json_outputs/M06_V010.json (deflated 83%)
  adding: kaggle/working/whisper_json_outputs/M06_V027.json (deflated 85%)
  adding: kaggle/working/whisper_json_outputs/M06_V015.json (deflated 79%)
  adding: kaggle/working/whisper_json_outputs/M06_V022.json (deflated 84%)
  adding: kaggle/working/whisper_json_outputs/M06_V005.json (deflated 81%)
  adding: kaggle/working/whisper_json_outputs/M06_V003.json (deflated 81%)
  adding: kaggle/working/whisper_json_outputs/M06_V025.json (deflated 80%)
  adding: kaggle/working/whisper_json_outputs/M06_V011.json (deflated 85%)
  adding: kaggle/working/whisper_json_out

/kaggle/working/M06.zip